# 01 数据整理与年度面板构造

本 Notebook 按作业要求完成以下工作：

- 遍历并检查原始数据结构
- 读取 CSMAR 原始表
- 统一为 `2010-2025` 年度年末口径
- 合并为一张年度面板表
- 构造作业要求变量
- 按作业 1.3 进行样本筛选
- 导出年度总表和筛选后样本

说明：

- 财务报表仅保留 `Typrep == 'A'` 的合并报表
- 财务报表仅保留 `12-31` 年报数据
- 为了按题意计算 2010 年 `Growth`，资产负债表会额外保留 2009 年年末总资产作为滞后缓冲
- `ST/PT` 标记使用 `LISTINGSTATE` 字段识别，凡状态含 `ST`、`PT` 或 `暂停` 均标记为异常上市状态
- `SOE` 按 `EquityNature` 是否含 `国企` 构造
- `NDTS` 主口径使用：固定资产折旧 + 无形资产摊销 + 长期待摊费用摊销


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("无法自动定位项目根目录，请在项目目录内运行 Notebook。")


ROOT = find_project_root()
RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
RAW_FILES = {
    "balance_sheet": RAW / "balance_sheet" / "balance_sheet.xlsx",
    "income_stmt": RAW / "income_stmt" / "income_stmt.xlsx",
    "cashflow": RAW / "cashflow" / "cashflow.xlsx",
    "ownership": RAW / "ownership" / "ownership.xlsx",
    "industry": RAW / "industry" / "industry.xlsx",
    "st_flag": RAW / "st_flag" / "st_flag.xlsx",
    "m2": RAW / "m2" / "m2.xlsx",
}

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT, RAW


(PosixPath('[项目根目录]'),
 PosixPath('[项目根目录]/data/raw'))

## 1. 遍历原始数据结构


In [2]:
raw_files = sorted(RAW.glob("*/*.xlsx"))
pd.DataFrame(
    {
        "file_name": [p.name for p in raw_files],
        "suffix": [p.suffix for p in raw_files],
        "size_mb": [round(p.stat().st_size / 1024 / 1024, 3) for p in raw_files],
    }
)


,file_name,suffix,size_mb
0,README_renaming.md,.md,0.002
1,balance_sheet.xlsx,.xlsx,38.783
2,balance_sheet_desc.txt,.txt,0.001
3,cashflow.xlsx,.xlsx,10.310
4,cashflow_desc.txt,.txt,0.001
5,income_stmt.xlsx,.xlsx,14.288
6,income_stmt_desc.txt,.txt,0.001
7,industry.xlsx,.xlsx,1.071
8,industry_desc.txt,.txt,0.001
9,m2.xlsx,.xlsx,0.005


In [3]:
def read_csmar_excel(path: Path) -> pd.DataFrame:
    return pd.read_excel(path, header=0, skiprows=[1, 2])

structure = {}
for name in [
    "balance_sheet",
    "income_stmt",
    "cashflow",
    "ownership",
    "industry",
    "st_flag",
    "m2",
]:
    df = read_csmar_excel(RAW_FILES[name])
    structure[name] = {
        "rows": len(df),
        "cols": len(df.columns),
        "columns": list(df.columns),
    }

pd.DataFrame(structure).T


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,rows,cols,columns
balance_sheet,535348,10,"[Stkcd, ShortName, Accper, Typrep, A001100000,..."
income_stmt,535305,5,"[Stkcd, ShortName, Accper, Typrep, B002000000]"
cashflow,223407,9,"[Stkcd, ShortName, Accper, Typrep, D000103000,..."
ownership,57211,4,"[Symbol, ShortName, EndDate, EquityNature]"
industry,58226,4,"[Symbol, ShortName, EndDate, IndustryCode]"
st_flag,58226,4,"[Symbol, ShortName, EndDate, LISTINGSTATE]"
m2,66,3,"[Staper, Efq0101, Efq0104]"


In [4]:
for name in ["balance_sheet", "income_stmt", "cashflow", "ownership", "industry", "st_flag", "m2"]:
    print(f"\n===== {name} =====")
    display(read_csmar_excel(RAW_FILES[name]).head())


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



===== balance_sheet =====


,Stkcd,ShortName,Accper,Typrep,A001100000,A001212000,A001200000,A001000000,A002100000,A002000000
0,1,深发展A,2009-12-31,A,NaN,1.714461e+09,NaN,5.878110e+11,NaN,5.673414e+11
1,1,深发展A,2010-01-01,A,NaN,1.714461e+09,NaN,5.878110e+11,NaN,5.673414e+11
2,1,深发展A,2010-03-31,A,NaN,1.918952e+09,NaN,6.199276e+11,NaN,5.978178e+11
3,1,深发展A,2010-06-30,A,NaN,2.133862e+09,NaN,6.243982e+11,NaN,5.939771e+11
4,1,深发展A,2010-09-30,A,NaN,2.403605e+09,NaN,6.750639e+11,NaN,6.429197e+11



===== income_stmt =====


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


...

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Symbol,ShortName,EndDate,LISTINGSTATE
0,1,深发展A,2009-12-31,正常上市
1,1,深发展A,2010-12-31,正常上市
2,1,深发展A,2011-12-31,正常上市
3,1,平安银行,2012-12-31,正常上市
4,1,平安银行,2013-12-31,正常上市



===== m2 =====


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Staper,Efq0101,Efq0104
0,2009-12,61022.45,28.50
1,2010-03,64994.75,22.49
2,2010-06,67392.17,18.46
3,2010-09,69647.15,18.97
4,2010-12,72585.18,19.73


## 2. 定义清洗函数


In [5]:
def normalize_code(series: pd.Series) -> pd.Series:
    return series.astype(str).str.extract(r"(\d+)")[0].str.zfill(6)


def filter_annual_year_end(df: pd.DataFrame, date_col: str, start: int = 2010, end: int = 2025) -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out = out.loc[out[date_col].dt.strftime("%m-%d").eq("12-31")].copy()
    out["year"] = out[date_col].dt.year
    out = out.loc[out["year"].between(start, end)].copy()
    return out


## 3. 读取并整理各张表


In [6]:
balance = read_csmar_excel(RAW_FILES["balance_sheet"])[
    [
        "Stkcd",
        "ShortName",
        "Accper",
        "Typrep",
        "A001100000",
        "A001212000",
        "A001000000",
        "A002100000",
        "A002000000",
    ]
].copy()

balance = filter_annual_year_end(balance, "Accper", start=2009, end=2025)
balance = balance.loc[balance["Typrep"].eq("A")].copy()
balance["stkcd"] = normalize_code(balance["Stkcd"])
balance["short_name"] = balance["ShortName"].astype(str)
balance = balance.rename(
    columns={
        "A001100000": "current_assets",
        "A001212000": "fixed_assets_net",
        "A001000000": "total_assets",
        "A002100000": "current_liabilities",
        "A002000000": "total_liabilities",
    }
)
balance = balance[
    [
        "stkcd",
        "short_name",
        "year",
        "current_assets",
        "fixed_assets_net",
        "total_assets",
        "current_liabilities",
        "total_liabilities",
    ]
].drop_duplicates(["stkcd", "year"], keep="last")

balance.head()


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,stkcd,short_name,year,current_assets,fixed_assets_net,total_assets,current_liabilities,total_liabilities
0,000001,深发展A,2009,NaN,1.714461e+09,5.878110e+11,NaN,5.673414e+11
5,000001,深发展A,2010,NaN,2.392293e+09,7.272071e+11,NaN,6.940095e+11
12,000001,深发展A,2011,NaN,3.524265e+09,1.258177e+12,NaN,1.182796e+12
22,000001,平安银行,2012,0.0,3.536443e+09,1.606537e+12,0.0,1.521738e+12
28,000001,平安银行,2013,0.0,3.694000e+09,1.891741e+12,0.0,1.779660e+12


In [7]:
income = read_csmar_excel(RAW_FILES["income_stmt"])[
    ["Stkcd", "Accper", "Typrep", "B002000000"]
].copy()

income = filter_annual_year_end(income, "Accper")
income = income.loc[income["Typrep"].eq("A")].copy()
income["stkcd"] = normalize_code(income["Stkcd"])
income = income.rename(columns={"B002000000": "net_profit"})
income = income[["stkcd", "year", "net_profit"]].drop_duplicates(["stkcd", "year"], keep="last")

income.head()


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,stkcd,year,net_profit
5,000001,2010,6.283816e+09
13,000001,2011,1.039049e+10
23,000001,2012,1.351078e+10
28,000001,2013,1.523100e+10
33,000001,2014,1.980200e+10


In [8]:
cashflow = read_csmar_excel(RAW_FILES["cashflow"])[
    [
        "Stkcd",
        "Accper",
        "Typrep",
        "D000103000",
        "D000119000",
        "D000120000",
        "D000104000",
        "D000105000",
    ]
].copy()

cashflow = filter_annual_year_end(cashflow, "Accper")
cashflow = cashflow.loc[cashflow["Typrep"].eq("A")].copy()
cashflow["stkcd"] = normalize_code(cashflow["Stkcd"])
cashflow = cashflow.rename(
    columns={
        "D000103000": "dep_fixed_assets",
        "D000119000": "dep_investment_property",
        "D000120000": "dep_rou_assets",
        "D000104000": "amort_intangible",
        "D000105000": "amort_long_term_deferred",
    }
)

for col in [
    "dep_fixed_assets",
    "dep_investment_property",
    "dep_rou_assets",
    "amort_intangible",
    "amort_long_term_deferred",
]:
    cashflow[col] = pd.to_numeric(cashflow[col], errors="coerce").fillna(0)

cashflow["ndts_numerator"] = (
    cashflow["dep_fixed_assets"]
    + cashflow["amort_intangible"]
    + cashflow["amort_long_term_deferred"]
)

cashflow = cashflow[
    [
        "stkcd",
        "year",
        "dep_fixed_assets",
        "dep_investment_property",
        "dep_rou_assets",
        "amort_intangible",
        "amort_long_term_deferred",
        "ndts_numerator",
    ]
].drop_duplicates(["stkcd", "year"], keep="last")

cashflow.head()


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,stkcd,year,dep_fixed_assets,dep_investment_property,dep_rou_assets,amort_intangible,amort_long_term_deferred,ndts_numerator
3,000001,2010,322211000.0,0.0,0.0,52458000.0,117783000.0,4.924520e+08
9,000001,2011,457021000.0,0.0,0.0,286890000.0,165949000.0,9.098600e+08
15,000001,2012,532397000.0,0.0,0.0,501503000.0,222466000.0,1.256366e+09
19,000001,2013,590000000.0,0.0,0.0,545000000.0,267000000.0,1.402000e+09
22,000001,2014,628000000.0,0.0,0.0,631000000.0,332000000.0,1.591000e+09


In [9]:
ownership = read_csmar_excel(RAW_FILES["ownership"])[
    ["Symbol", "EndDate", "EquityNature"]
].copy()

ownership = filter_annual_year_end(ownership, "EndDate")
ownership["stkcd"] = normalize_code(ownership["Symbol"])
ownership["equity_nature"] = ownership["EquityNature"].astype(str)
ownership["soe"] = ownership["equity_nature"].str.contains("国企", na=False).astype("Int64")
ownership = ownership[["stkcd", "year", "equity_nature", "soe"]].drop_duplicates(["stkcd", "year"], keep="last")

ownership.head()


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,stkcd,year,equity_nature,soe
1,000001,2010,外资,0
2,000001,2011,外资,0
3,000001,2012,外资,0
4,000001,2013,外资,0
5,000001,2014,外资,0


In [10]:
industry = read_csmar_excel(RAW_FILES["industry"])[
    ["Symbol", "EndDate", "IndustryCode"]
].copy()

industry = filter_annual_year_end(industry, "EndDate")
industry["stkcd"] = normalize_code(industry["Symbol"])
industry["industry_code"] = industry["IndustryCode"].astype(str).str.strip()
industry["industry_group"] = np.where(
    industry["industry_code"].str.startswith("C"),
    industry["industry_code"].str[:3],
    industry["industry_code"].str[:1],
)
industry = industry[["stkcd", "year", "industry_code", "industry_group"]].drop_duplicates(["stkcd", "year"], keep="last")

industry.head()


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,stkcd,year,industry_code,industry_group
1,000001,2010,I0199,I
2,000001,2011,I0199,I
3,000001,2012,J66,J
4,000001,2013,J66,J
5,000001,2014,J66,J


In [11]:
st_flag = read_csmar_excel(RAW_FILES["st_flag"])[
    ["Symbol", "EndDate", "LISTINGSTATE"]
].copy()

st_flag = filter_annual_year_end(st_flag, "EndDate")
st_flag["stkcd"] = normalize_code(st_flag["Symbol"])
st_flag["listing_state"] = st_flag["LISTINGSTATE"].astype(str).str.strip()
st_flag["is_stpt_year"] = st_flag["listing_state"].str.contains(r"ST|PT|暂停", na=False).astype("Int64")
st_flag["ever_stpt"] = st_flag.groupby("stkcd")["is_stpt_year"].transform("max").astype("Int64")
st_flag = st_flag[["stkcd", "year", "listing_state", "is_stpt_year", "ever_stpt"]].drop_duplicates(["stkcd", "year"], keep="last")

st_flag["listing_state"].value_counts(dropna=False)


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


listing_state
正常上市    54752
*ST       885
ST        716
暂停上市      121
Name: count, dtype: int64

In [12]:
m2 = read_csmar_excel(RAW_FILES["m2"])[
    ["Staper", "Efq0104"]
].copy()

m2["Staper"] = pd.to_datetime(m2["Staper"], errors="coerce")
m2 = m2.loc[m2["Staper"].dt.month.eq(12)].copy()
m2["year"] = m2["Staper"].dt.year
m2 = m2.loc[m2["year"].between(2010, 2025)].copy()
m2["m2_growth"] = pd.to_numeric(m2["Efq0104"], errors="coerce")
m2 = m2[["year", "m2_growth"]].drop_duplicates(["year"], keep="last")

m2


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,year,m2_growth
4,2010,19.73
8,2011,13.61
12,2012,13.84
16,2013,13.59
20,2014,12.16
24,2015,13.34
28,2016,11.30
32,2017,8.07
36,2018,8.10
40,2019,8.70


## 4. 合并年度面板


In [13]:
panel = balance.merge(income, on=["stkcd", "year"], how="outer")
panel = panel.merge(cashflow, on=["stkcd", "year"], how="outer")
panel = panel.merge(ownership, on=["stkcd", "year"], how="left")
panel = panel.merge(industry, on=["stkcd", "year"], how="left")
panel = panel.merge(st_flag, on=["stkcd", "year"], how="left")
panel = panel.merge(m2, on="year", how="left")

panel = panel.sort_values(["stkcd", "year"]).reset_index(drop=True)
panel.head()


,stkcd,short_name,year,current_assets,fixed_assets_net,total_assets,current_liabilities,total_liabilities,net_profit,dep_fixed_assets,dep_investment_property,dep_rou_assets,amort_intangible,amort_long_term_deferred,ndts_numerator,equity_nature,soe,industry_code,industry_group,listing_state,is_stpt_year,ever_stpt,m2_growth
0,000001,深发展A,2009,NaN,1.714461e+09,5.878110e+11,NaN,5.673414e+11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,<NA>,<NA>,NaN
1,000001,深发展A,2010,NaN,2.392293e+09,7.272071e+11,NaN,6.940095e+11,6.283816e+09,322211000.0,0.0,0.0,52458000.0,117783000.0,4.924520e+08,外资,0,I0199,I,正常上市,0,0,19.73
2,000001,深发展A,2011,NaN,3.524265e+09,1.258177e+12,NaN,1.182796e+12,1.039049e+10,457021000.0,0.0,0.0,286890000.0,165949000.0,9.098600e+08,外资,0,I0199,I,正常上市,0,0,13.61
3,000001,平安银行,2012,0.0,3.536443e+09,1.606537e+12,0.0,1.521738e+12,1.351078e+10,532397000.0,0.0,0.0,501503000.0,222466000.0,1.256366e+09,外资,0,J66,J,正常上市,0,0,13.84
4,000001,平安银行,2013,0.0,3.694000e+09,1.891741e+12,0.0,1.779660e+12,1.523100e+10,590000000.0,0.0,0.0,545000000.0,267000000.0,1.402000e+09,外资,0,J66,J,正常上市,0,0,13.59


## 5. 按作业要求构造变量


In [14]:
numeric_cols = [
    "current_assets",
    "fixed_assets_net",
    "total_assets",
    "current_liabilities",
    "total_liabilities",
    "net_profit",
    "dep_fixed_assets",
    "dep_investment_property",
    "dep_rou_assets",
    "amort_intangible",
    "amort_long_term_deferred",
    "ndts_numerator",
]

for col in numeric_cols:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

panel["lev"] = panel["total_liabilities"] / panel["total_assets"]
panel["npr"] = panel["net_profit"] / panel["total_assets"]
panel["size"] = np.where(panel["total_assets"] > 0, np.log(panel["total_assets"]), np.nan)
panel["tang"] = panel["fixed_assets_net"] / panel["total_assets"]
panel["growth"] = (
    panel["total_assets"] - panel.groupby("stkcd")["total_assets"].shift(1)
) / panel.groupby("stkcd")["total_assets"].shift(1)
panel["ndts"] = panel["ndts_numerator"] / panel["total_assets"]
panel["liq"] = panel["current_assets"] / panel["current_liabilities"]

panel[
    [
        "stkcd",
        "year",
        "lev",
        "npr",
        "size",
        "tang",
        "growth",
        "ndts",
        "liq",
        "soe",
        "industry_code",
        "listing_state",
        "ever_stpt",
        "m2_growth",
    ]
].head()


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,stkcd,year,lev,npr,size,tang,growth,ndts,liq,soe,industry_code,listing_state,ever_stpt,m2_growth
0,000001,2009,0.965177,NaN,27.099671,0.002917,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN
1,000001,2010,0.954349,0.008641,27.312477,0.003290,0.237144,0.000677,NaN,0,I0199,正常上市,0,19.73
2,000001,2011,0.940087,0.008258,27.860685,0.002801,0.730149,0.000723,NaN,0,I0199,正常上市,0,13.61
3,000001,2012,0.947216,0.008410,28.105102,0.002201,0.276877,0.000782,NaN,0,J66,正常上市,0,13.84
4,000001,2013,0.940752,0.008051,28.268519,0.001953,0.177527,0.000741,NaN,0,J66,正常上市,0,13.59


In [15]:
panel["ind_code"] = panel["industry_group"]

manuf_counts = panel.loc[
    panel["industry_group"].fillna("").str.startswith("C"),
    "industry_group"
].value_counts()
small_manuf_groups = manuf_counts[manuf_counts < 30].index
panel.loc[panel["industry_group"].isin(small_manuf_groups), "ind_code"] = "C_other"

panel["ind_code"].value_counts().head(20)


ind_code
C39    5384
I      3918
C26    3443
C35    3320
C38    3308
C27    3100
F      2412
C34    2053
C36    1796
K      1760
G      1689
D      1619
J      1477
E      1293
C30    1285
C29    1188
B      1130
C32     937
C33     935
M       917
Name: count, dtype: int64

## 6. 只保留作业会用到的变量


In [16]:
panel_final = panel[
    [
        "stkcd",
        "short_name",
        "year",
        "lev",
        "npr",
        "size",
        "tang",
        "growth",
        "ndts",
        "liq",
        "soe",
        "equity_nature",
        "industry_code",
        "ind_code",
        "listing_state",
        "is_stpt_year",
        "ever_stpt",
        "m2_growth",
    ]
].sort_values(["stkcd", "year"]).reset_index(drop=True)

panel_final = panel_final.loc[panel_final["year"].between(2010, 2025)].copy()

panel_final.head()


,stkcd,short_name,year,lev,npr,size,tang,growth,ndts,liq,soe,equity_nature,industry_code,ind_code,listing_state,is_stpt_year,ever_stpt,m2_growth
1,000001,深发展A,2010,0.954349,0.008641,27.312477,0.003290,0.237144,0.000677,NaN,0,外资,I0199,I,正常上市,0,0,19.73
2,000001,深发展A,2011,0.940087,0.008258,27.860685,0.002801,0.730149,0.000723,NaN,0,外资,I0199,I,正常上市,0,0,13.61
3,000001,平安银行,2012,0.947216,0.008410,28.105102,0.002201,0.276877,0.000782,NaN,0,外资,J66,J,正常上市,0,0,13.84
4,000001,平安银行,2013,0.940752,0.008051,28.268519,0.001953,0.177527,0.000741,NaN,0,外资,J66,J,正常上市,0,0,13.59
5,000001,平安银行,2014,0.940109,0.009057,28.413304,0.001743,0.155792,0.000728,NaN,0,外资,J66,J,正常上市,0,0,12.16


In [17]:
panel_final.info()


<class 'pandas.DataFrame'>
Index: 57605 entries, 1 to 59361
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   stkcd          57605 non-null  str    
 1   short_name     57602 non-null  str    
 2   year           57605 non-null  int32  
 3   lev            57605 non-null  float64
 4   npr            57605 non-null  float64
 5   size           57604 non-null  float64
 6   tang           57601 non-null  float64
 7   growth         53688 non-null  float64
 8   ndts           57600 non-null  float64
 9   liq            56547 non-null  float64
 10  soe            55457 non-null  Int64  
 11  equity_nature  53857 non-null  str    
 12  industry_code  56471 non-null  str    
 13  ind_code       56471 non-null  str    
 14  listing_state  56471 non-null  str    
 15  is_stpt_year   56471 non-null  Int64  
 16  ever_stpt      56471 non-null  Int64  
 17  m2_growth      57605 non-null  float64
dtypes: Int64(3), float64(8

In [18]:
panel_final.describe(include="all").T.head(20)


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/pandas/core/nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/pandas/core/nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/pandas/core/nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/pandas/core/nanops.py:1027: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
stkcd,57605,5671,000001,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
short_name,57602,7832,三维股份,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,57605.0,NaN,NaN,NaN,2018.55089,4.280055,2010.0,2015.0,2019.0,2022.0,2025.0
lev,57605.0,NaN,NaN,NaN,inf,NaN,-0.194698,0.250494,0.41207,0.584017,inf
npr,57605.0,NaN,NaN,NaN,inf,NaN,-48.315918,0.010691,0.035671,0.067502,inf
size,57604.0,NaN,NaN,NaN,22.161253,1.586741,13.075966,21.147446,21.940041,22.935611,31.610287
tang,57601.0,NaN,NaN,NaN,0.200091,0.159169,0.0,0.076263,0.166445,0.288589,0.970921
growth,53688.0,NaN,NaN,NaN,inf,NaN,-1.0,-0.00295,0.075113,0.189998,inf
ndts,57600.0,NaN,NaN,NaN,inf,NaN,-0.018391,0.011269,0.020381,0.032066,inf
liq,56547.0,NaN,NaN,NaN,2.802696,4.614437,-5.131645,1.177087,1.74728,2.970442,552.479345


## 7. 按作业 1.3 执行样本筛选


In [19]:
sample_steps = []
filtered = panel_final.copy()

sample_steps.append({
    "step": "初始样本",
    "dropped_obs": np.nan,
    "remaining_obs": len(filtered),
    "remaining_firms": filtered["stkcd"].nunique(),
})

before = len(filtered)
filtered = filtered.loc[~filtered["industry_code"].astype(str).str.startswith("J")].copy()
sample_steps.append({
    "step": "剔除金融保险",
    "dropped_obs": before - len(filtered),
    "remaining_obs": len(filtered),
    "remaining_firms": filtered["stkcd"].nunique(),
})

before = len(filtered)
filtered = filtered.loc[filtered["ever_stpt"] != 1].copy()
sample_steps.append({
    "step": "剔除ST/PT",
    "dropped_obs": before - len(filtered),
    "remaining_obs": len(filtered),
    "remaining_firms": filtered["stkcd"].nunique(),
})

before = len(filtered)
filtered = filtered.loc[filtered["lev"] <= 1].copy()
sample_steps.append({
    "step": "剔除Lev > 1",
    "dropped_obs": before - len(filtered),
    "remaining_obs": len(filtered),
    "remaining_firms": filtered["stkcd"].nunique(),
})

key_vars = ["lev", "npr", "size", "tang", "growth", "ndts", "soe", "ind_code", "m2_growth"]
before = len(filtered)
filtered = filtered.dropna(subset=key_vars).copy()
sample_steps.append({
    "step": "剔除缺失值",
    "dropped_obs": before - len(filtered),
    "remaining_obs": len(filtered),
    "remaining_firms": filtered["stkcd"].nunique(),
})

sample_selection = pd.DataFrame(sample_steps)
sample_selection


,step,dropped_obs,remaining_obs,remaining_firms
0,初始样本,NaN,57605,5671
1,剔除金融保险,1477.0,56128,5616
2,剔除ST/PT,10857.0,45271,4888
3,剔除Lev > 1,20.0,45251,4888
4,剔除缺失值,4405.0,40846,4775


In [20]:
filtered.head()


,stkcd,short_name,year,lev,npr,size,tang,growth,ndts,liq,soe,equity_nature,industry_code,ind_code,listing_state,is_stpt_year,ever_stpt,m2_growth
1,000001,深发展A,2010,0.954349,0.008641,27.312477,0.003290,0.237144,0.000677,NaN,0,外资,I0199,I,正常上市,0,0,19.73
2,000001,深发展A,2011,0.940087,0.008258,27.860685,0.002801,0.730149,0.000723,NaN,0,外资,I0199,I,正常上市,0,0,13.61
20,000002,万科A,2012,0.783163,0.041348,26.660278,0.004256,0.278835,0.000482,1.396177,1,国企,K70,K,正常上市,0,0,13.84
21,000002,万科A,2013,0.779970,0.038183,26.895395,0.004444,0.265056,0.000374,1.343926,1,国企,K70,K,正常上市,0,0,13.59
22,000002,万科A,2014,0.772046,0.037937,26.954552,0.004540,0.060941,0.001080,1.344714,1,国企,K70,K,正常上市,0,0,12.16


## 8. 导出 clean 数据


In [21]:
CLEAN.mkdir(parents=True, exist_ok=True)

csv_path = CLEAN / "panel_capital_structure_annual.csv"
parquet_path = CLEAN / "panel_capital_structure_annual.parquet"
summary_path = CLEAN / "panel_capital_structure_annual_summary.md"
filtered_csv_path = CLEAN / "panel_capital_structure_filtered.csv"
filtered_parquet_path = CLEAN / "panel_capital_structure_filtered.parquet"
sample_selection_path = CLEAN / "sample_selection_013.csv"

panel_final.to_csv(csv_path, index=False, encoding="utf-8-sig")
panel_final.to_parquet(parquet_path, index=False)
filtered.to_csv(filtered_csv_path, index=False, encoding="utf-8-sig")
filtered.to_parquet(filtered_parquet_path, index=False)
sample_selection.to_csv(sample_selection_path, index=False, encoding="utf-8-sig")

summary = [
    "# Panel Build Summary",
    "",
    f"- 行数: {len(panel_final):,}",
    f"- 公司数: {panel_final['stkcd'].nunique():,}",
    f"- 年份范围: {int(panel_final['year'].min())}-{int(panel_final['year'].max())}",
    f"- ever_stpt=1 的公司数: {panel_final.loc[panel_final['ever_stpt'].eq(1), 'stkcd'].nunique():,}",
]

summary_path.write_text("\n".join(summary), encoding="utf-8")

print(csv_path)
print(parquet_path)
print(filtered_csv_path)
print(filtered_parquet_path)
print(sample_selection_path)
print(summary_path)


[项目根目录]/data/clean/panel_capital_structure_annual.csv
[项目根目录]/data/clean/panel_capital_structure_annual.parquet
[项目根目录]/data/clean/panel_capital_structure_filtered.csv
[项目根目录]/data/clean/panel_capital_structure_filtered.parquet
[项目根目录]/data/clean/sample_selection_013.csv
[项目根目录]/data/clean/panel_capital_structure_annual_summary.md


In [22]:
panel_final.tail()


,stkcd,short_name,year,lev,npr,size,tang,growth,ndts,liq,soe,equity_nature,industry_code,ind_code,listing_state,is_stpt_year,ever_stpt,m2_growth
59357,920992,中科美菱,2021,0.534924,0.129837,20.069019,0.216057,0.014789,0.016575,1.432640,<NA>,NaN,C35,C35,正常上市,0,0,9.0
59358,920992,中科美菱,2022,0.255193,0.065153,20.508990,0.162593,0.552662,0.013174,3.413553,1,国企,C35,C35,正常上市,0,0,11.8
59359,920992,中科美菱,2023,0.179175,0.022006,20.412416,0.165643,-0.092058,0.017436,4.888089,1,国企,C35,C35,正常上市,0,0,9.7
59360,920992,中科美菱,2024,0.179176,0.024269,20.429266,0.149150,0.016993,0.018387,4.912023,1,国企,C35,C35,正常上市,0,0,7.3
59361,920992,中科美菱,2025,0.166527,0.025478,20.435496,0.133151,0.006250,0.021303,5.167978,1,国企,C35,C35,正常上市,0,0,8.5
